# ЛР-4. Сбор данных с динамических страниц: Playwright и no-code — анализ результатов

**Студент:** Подольский Ф. А., ПМИ-3-2025 · **Вариант 5:** `quotes.toscrape.com/login` + разделы за входом
(форма с CSRF, `storage_state`, пагинация после авторизации); контрольный статический источник — `quotes.toscrape.com/page/1`.

Сбор выполняют скрипты (`run_real.sh` запускает всё по порядку):

| Скрипт | Что делает | Что пишет |
|---|---|---|
| `diagnose.py` | протокол диагностики источника | `metrics/diagnostics.json` |
| `collect.py` | браузерный сбор (Playwright, sync API) | `data_browser.csv`, `quotes_browser.csv`, `collection_log.csv`, `auth_state.json`, `metrics/browser.json`, `artifacts/` |
| `light.py` | дешёвая альтернатива (requests.Session + CSRF) | `data_light.csv`, `quotes_light.csv`, `metrics/light.json` |
| `compare.py` | сверка наборов | `metrics/compare.json` |
| `bench.py` | замеры оптимизаций | `metrics/bench_*.csv`, `metrics/bench.json` |
| `faults.py`, `parallel.py` | бонусы: подмена ответов, параллельный сбор | `metrics/faults.json`, `metrics/parallel.json`, `artifacts/trace_faults.zip` |
| `summarize.py` | таблицы для отчёта | `metrics/summary.md` |

Блокнот **только анализирует** готовые файлы и один раз обращается к сайту — чтобы проверить, что сохранённая сессия
(`auth_state.json`) действует. В Jupyter уже работает цикл событий asyncio, поэтому здесь используется
**асинхронный** API Playwright (`await`), а в `.py`-скриптах — синхронный.

In [ ]:
import json, os, sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

LAB = Path.cwd()
if not (LAB / "common.py").exists():           # блокнот запущен не из папки лабораторной
    LAB = Path(os.getenv("LAB_DIR", ".")).resolve()
sys.path.insert(0, str(LAB))
import common as C
import summarize as S
import compare as CMP

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

# палитра: два первых категориальных слота эталонной палитры, текст — нейтральный
BLUE, ORANGE, INK, MUTED, SURFACE = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": MUTED,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
                     "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True,
                     "grid.color": "#e4e3df", "grid.linewidth": 0.6,
                     "axes.spines.top": False, "axes.spines.right": False})

print("BASE_URL :", C.BASE_URL)
print("Данные   :", C.OUT_DIR)
if C.BASE_URL != "https://quotes.toscrape.com":
    print("ВНИМАНИЕ: тестовый BASE_URL — результаты НЕ с реального сайта, в отчёт не переносить.")

## 1. Протокол диагностики источника

Каждая строка — результат измерения `diagnose.py` (requests, Playwright, Playwright с отключённым JS).

In [ ]:
diag = json.loads((C.METRICS / "diagnostics.json").read_text(encoding="utf-8"))
display(S.diagnostics_table())
print("Флаги вывода:", {k: v for k, v in diag["conclusion"].items() if k != "text"})

## 2. Сохранённая сессия: проверка `auth_state.json` (async API)

Новый контекст создаётся сразу авторизованным из `storage_state` — повторный вход не нужен.
Признаки сессии: ссылка **Logout** в шапке и ссылки **«(Goodreads page)»**, которые сервер показывает только после входа.

In [ ]:
from playwright.async_api import async_playwright


async def check_state() -> dict:
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True, executable_path=C.CHROMIUM_PATH)
        try:
            ctx = await browser.new_context(storage_state=str(C.STATE_PATH), user_agent=C.USER_AGENT,
                                            locale="en-US", viewport={"width": 1280, "height": 900})
            ctx.set_default_timeout(15_000)
            ctx.set_default_navigation_timeout(30_000)
            page = await ctx.new_page()
            await page.goto(f"{C.BASE_URL}/page/1/", wait_until="domcontentloaded")
            await page.wait_for_selector(C.QUOTE_SEL, state="attached")   # ожидание, не sleep
            res = {"logout_link": await page.get_by_role("link", name="Logout", exact=True).count() > 0,
                   "cards": await page.locator(C.QUOTE_SEL).count(),
                   "goodreads_links": await page.locator(C.QUOTE_SEL).locator('a[href*="goodreads.com"]').count()}
            await ctx.close()
            return res
        finally:
            await browser.close()


if C.STATE_PATH.exists():
    state = json.loads(C.STATE_PATH.read_text(encoding="utf-8"))
    print("Cookies в auth_state.json:", [(c["name"], C.mask(c["value"], 4), c["domain"]) for c in state["cookies"]])
    chk = await check_state()
    print("Проверка сессии:", chk)
    print("Сессия действует — новый контекст авторизован без входа" if chk["logout_link"] else
          "Сессия истекла — при следующем запуске collect.py выполнит вход один раз и обновит auth_state.json")
else:
    print("auth_state.json нет — сначала запустите collect.py")

## 3. Объём набора и журнал сбора

Единица итогового набора — **«цитата в разделе тега»**: на сайте 100 цитат (10 страниц ленты), поэтому объём
≥ 200 набирается обходом разделов тегов `/tag/<тег>/page/N` после входа. Бизнес-ключ записи —
`record_key = sha1(тег | quote_id)`, где `quote_id = sha1(нормализованный текст | автор)`: одна и та же цитата
в одном теге не может попасть в набор дважды (повторный обход, перекрытие страниц), а в разных тегах — это разные
факты принадлежности. Число **уникальных** цитат показано отдельно, чтобы объём не выглядел больше, чем он есть.

In [ ]:
display(S.volume_table())
display(S.log_summary())

In [ ]:
log = pd.read_csv(C.LOG_PATH, keep_default_na=False)
lb = log[(log["approach"] == "browser") & (log["stage"] != "auth")].copy()
for c in ("iteration", "new_records", "duration_s"):
    lb[c] = pd.to_numeric(lb[c], errors="coerce")
print("Статусы итераций браузера:", lb["status"].value_counts().to_dict())
print("Причины остановки разделов (browser):",
      pd.Series(json.loads((C.METRICS / "browser.json").read_text(encoding="utf-8"))["stop_reasons"]).value_counts().to_dict())

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for stage, color in (("feed", BLUE), ("tag", ORANGE)):
    part = lb[lb["stage"] == stage]
    ax[0].bar(part["iteration"], part["new_records"], color=color, width=0.8, label=stage)
    ax[1].bar(part["iteration"], part["duration_s"], color=color, width=0.8, label=stage)
ax[0].set_title("Прирост записей за итерацию (браузер)", loc="left")
ax[1].set_title("Длительность итерации, с (браузер)", loc="left")
for a in ax:
    a.set_xlabel("итерация")
    a.legend(frameon=False)
plt.tight_layout()
plt.show()

## 4. Качество данных и smoke-check

Обязательные поля: `text`, `author`, `author_url`, `goodreads_url` (появляется только после входа — это заодно
проверка, что сессия была активна), `tag`. Порог доли пустых — 5 %.

In [ ]:
db = C.typed(pd.read_csv(C.OUT_DIR / "data_browser.csv", keep_default_na=False))
dl = C.typed(pd.read_csv(C.OUT_DIR / "data_light.csv", keep_default_na=False))
qb = C.quality_report(db)
print("Записей:", qb["records"], "| уникальных цитат:", qb["unique_quotes"], "| тегов:", qb["unique_tags"],
      "| авторов:", qb["unique_authors"], "| дубликатов по ключу:", qb["duplicates_by_key"])
print("Доля пустых:", qb["empty_share"])
display(pd.Series(qb["dtypes"], name="dtype").to_frame().T)
display(S.quality_table())
display(db.head(5))

In [ ]:
# Демонстрация, что smoke-check действительно срабатывает: имитируем потерю сессии у 10 % строк
broken = db.copy()
broken.loc[broken.sample(frac=0.10, random_state=5).index, "goodreads_url"] = ""
warnings = C.smoke_check(broken, name="data_browser (искусственная порча)")
assert warnings, "smoke-check должен был выдать предупреждение"

## 5. Сверка: браузер ↔ requests

Сравниваются множества бизнес-ключей и значения полей по общим ключам; отдельно — согласованность
«теги из метаданных ленты» ↔ «найдено на страницах тегов».

In [ ]:
res = {"pairs": CMP.diff_sets(db, dl, "record_key", "browser", "light", CMP.FIELDS + ["tag"])}
qbr = C.typed(pd.read_csv(C.OUT_DIR / "quotes_browser.csv", keep_default_na=False))
qli = C.typed(pd.read_csv(C.OUT_DIR / "quotes_light.csv", keep_default_na=False))
res["feed"] = CMP.diff_sets(qbr, qli, "quote_id", "browser", "light", CMP.FIELDS)
for k, r in res.items():
    print(f"{k}: браузер={r['browser_records']} requests={r['light_records']} общих={r['common_keys']} "
          f"только браузер={r['only_browser_count']} только requests={r['only_light_count']} "
          f"расхождений полей={ {f: v['count'] for f, v in r['field_mismatches'].items()} } "
          f"→ {'совпадают' if r['identical'] else 'ЕСТЬ РАСХОЖДЕНИЯ'}")
    for f, v in r["field_mismatches"].items():
        display(pd.DataFrame(v["examples"]))
print("\nСогласованность (браузер):", CMP.consistency(qbr, db))

## 6. Таблица сравнения подходов (измеренные значения)

Строки с маркером ⟪ЗАПОЛНИТЬ⟫ заполняются вручную: время разработки — по своему учёту, no-code — по экрану
Executions в n8n и «Мониторингу системы».

In [ ]:
display(S.comparison_table())

## 7. Эффект оптимизаций браузерного режима

In [ ]:
bench = json.loads((C.METRICS / "bench.json").read_text(encoding="utf-8"))
bb = pd.read_csv(C.METRICS / "bench_block.csv")
be = pd.read_csv(C.METRICS / "bench_extract.csv")
ba = pd.read_csv(C.METRICS / "bench_auth.csv")
display(bb.groupby("mode")[["seconds", "requests", "blocked", "kb"]].median())
print("Экономия от блокировки: время", bench["block"].get("time_saving_pct"), "%, трафик",
      bench["block"].get("traffic_saving_pct"), "%")
print("Извлечение:", bench["extract"])
print("Авторизация:", bench["auth"])

med = bb.groupby("mode")[["seconds", "kb"]].median().reindex(["full", "block"])
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
labels = ["всё грузим", "блокируем"]
ax[0].bar(labels, med["seconds"], color=[BLUE, ORANGE], width=0.5)
ax[0].set_title(f"Лента {bench['pages']} стр.: время, с (медиана)", loc="left")
ax[1].bar(labels, med["kb"], color=[BLUE, ORANGE], width=0.5)
ax[1].set_title("Трафик, КБ (медиана)", loc="left")
x = be["page"]
ax[2].bar(x - 0.2, be["bulk_ms"], width=0.4, color=BLUE, label="eval_on_selector_all")
ax[2].bar(x + 0.2, be["per_element_ms"], width=0.4, color=ORANGE, label="поэлементно")
ax[2].set_title("Извлечение страницы, мс", loc="left")
ax[2].set_xlabel("страница ленты")
ax[2].legend(frameon=False, loc="upper left", bbox_to_anchor=(0, -0.2), ncol=2)
plt.tight_layout()
plt.show()
display(ba)

## 8. Задания повышенной сложности

In [ ]:
for name in ("parallel.json", "faults.json"):
    p = C.METRICS / name
    if not p.exists():
        print(name, "— нет (скрипт не запускался)")
        continue
    data = json.loads(p.read_text(encoding="utf-8"))
    if name == "parallel.json":
        print(f"Параллельный сбор: тегов {data['tags']}, K=1 {data['sequential']['seconds']} c, "
              f"K={data['parallel']['workers']} {data['parallel']['seconds']} c, ускорение ×{data['speedup_x']}, "
              f"ошибок {len(data['sequential']['errors']) + len(data['parallel']['errors'])}, "
              f"наборы совпадают: {data['same_records_seq_vs_par']}, совпадают с data_browser: {data['same_as_data_browser']}")
    else:
        display(pd.DataFrame(data["checks"]))
print("Трассы:", sorted(p.name for p in C.ARTIFACTS.glob("*.zip")), "→ открыть: playwright show-trace artifacts/trace_faults.zip")

## 9. No-code (n8n)

Результат workflow — `nocode/nocode_result.csv` (лента после входа, 1 строка = цитата). Сверяется с лентой браузера по `quote_id`.

In [ ]:
nc_path = C.OUT_DIR / "nocode" / "nocode_result.csv"
if nc_path.exists():
    nc = pd.read_csv(nc_path, keep_default_na=False)
    print("Строк:", len(nc), "| столбцы:", list(nc.columns))
    print("Сверка с лентой браузера:", CMP.nocode_check(qbr))
    C.smoke_check(nc.rename(columns=str.lower), required=["text", "author", "author_url", "goodreads_url"], name="nocode_result.csv")
    display(nc.head(5))
else:
    print("nocode/nocode_result.csv ещё нет — выполните workflow по nocode/INSTRUCTIONS.md")

## Выводы

Итоговые выводы, рекомендации по трём сценариям и правовая оценка — в `report.md`; таблицы для него
собирает `summarize.py` (`metrics/summary.md`).